In [0]:
%sql
DELETE FROM dbjahan.bronze_data.folkmangd_raw;

num_affected_rows
12480


In [0]:
%sql
USE dbjahan.bronze_data;

CREATE OR REPLACE TABLE dbjahan.bronze_data.folkmangd_raw (
  Region STRING,
  Alder STRING,
  Kon STRING,
  Tid STRING,
  Folkmangd BIGINT,
  Civilstand STRING
)
USING DELTA;

In [0]:
import requests
import itertools
import time
from pyspark.sql import Row

def hamta(api_namn, url, params=None, timeout=90):
    senaste = None
    for forsok in range(1, 4):
        try:
            svar = requests.get(url, params=params, timeout=timeout)
            if svar.status_code == 200:
                return svar
            senaste = str(svar.status_code)
        except requests.RequestException as fel:
            senaste = str(fel)
        if forsok < 3:
            time.sleep(5)
    raise RuntimeError(
        f"{api_namn} svarade {senaste} efter 3 försök. URL: {url}"
    )

url = "https://statistikdatabasen.scb.se/api/v2/tables/TAB638/data"
params = {
    "lang": "sv",
    "outputFormat": "json-stat2",
    "valueCodes[ContentsCode]": "BE0101N1",
    "valueCodes[Region]": "*",
    "valueCodes[Alder]": "tot",
    "valueCodes[Civilstand]": "*",
    "valueCodes[Kon]": "*",
    "valueCodes[Tid]": "from(2020)",
}

svar = hamta("SCB TAB638", url, params=params, timeout=90)
js = svar.json()

ids = js["id"]
dims = []
for var in ids:
    cat = js["dimension"][var]["category"]
    index = cat["index"]
    labels = cat.get("label", {})
    codes = [None] * len(index)
    for kod, plats in index.items():
        codes[plats] = (kod, labels.get(kod, kod))
    dims.append(codes)

rader = []
for i, combo in enumerate(itertools.product(*dims)):
    d = {var: kod for var, (kod, _txt) in zip(ids, combo)}
    val = js["value"][i]
    rader.append(Row(
        Region=d["Region"],
        Alder=d["Alder"],
        Civilstand=d["Civilstand"],
        Kon=d["Kon"],
        Tid=d["Tid"],
        Folkmangd=int(val) if val is not None else None,
    ))

df = spark.createDataFrame(rader)
df.createOrReplaceTempView("folk_nya")

spark.sql("""
INSERT INTO dbjahan.bronze_data.folkmangd_raw
  (Region, Alder, Civilstand, Kon, Tid, Folkmangd)
SELECT n.Region, n.Alder, n.Civilstand, n.Kon, n.Tid, n.Folkmangd
FROM folk_nya n
WHERE NOT EXISTS (
  SELECT 1
  FROM dbjahan.bronze_data.folkmangd_raw t
  WHERE t.Region = n.Region
    AND t.Alder = n.Alder
    AND t.Civilstand = n.Civilstand
    AND t.Kon = n.Kon
    AND t.Tid = n.Tid
)
""")
print("Klart folkmangd", df.count())

Klart folkmangd 12480


In [0]:
%sql
SELECT * FROM dbjahan.bronze_data.folkmangd_raw
LIMIT 20;

Region,Alder,Kon,Tid,Folkmangd
1438,tot,1,2020,1480
1438,tot,1,2021,1476
1438,tot,1,2022,1446
1438,tot,1,2023,1435
1438,tot,1,2024,1445
1438,tot,2,2020,1090
1438,tot,2,2021,1091
1438,tot,2,2022,1074
1438,tot,2,2023,1076
1438,tot,2,2024,1070


In [0]:
%sql
SELECT * FROM dbjahan.bronze_data.folkmangd_raw

Region,Alder,Kon,Tid,Folkmangd
1499,tot,1,2020,9496
1499,tot,1,2021,9548
1499,tot,1,2022,9526
1499,tot,1,2023,9463
1499,tot,1,2024,9372
1499,tot,2,2020,8001
1499,tot,2,2021,8035
1499,tot,2,2022,8037
1499,tot,2,2023,7971
1499,tot,2,2024,7960
